# Showing uncertainty & telling the story

_Data Wrangling_

**Present a finding honestly: show the error bars, and make one clear point per chart.**

You have done the wrangling. The data is clean, the analysis is done, and you have a number:
       group A averages 4.2, group B averages 3.7. The temptation is to draw two bars and
       declare A the winner. That is where honest communication begins &mdash; and where most reports go
       wrong.

       Two jobs sit on top of every results figure. First, show the uncertainty. Your 4.2 came
       from a sample, not the whole population, so it is fuzzy. How fuzzy? A 95% confidence
       interval (CI) answers that: a range that would contain the true mean about 95 times out of 100
       if you repeated the study. Draw it as an error bar on each bar, a shaded band around
       a line, or by showing the whole distribution instead of a lone summary. If A's bar reads
       4.2 &plusmn; 0.1 and B's reads 3.7 &plusmn; 0.1, the gap is real; if both bars are &plusmn; 0.6,
       you cannot tell them apart.

---

This notebook builds an honest results chart one piece at a time. Run each cell, read the note above it, then tackle the practice problems at the end. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — matplotlib + numpy + pandas

We take a small A/B result and turn it into an honest, story-telling chart in three steps: (1) make the data, (2) compute each group's mean and 95% confidence interval, (3) plot the means with error bars and the annotations that state the finding.

### Step 1 — Build a small A/B result

We simulate a tiny experiment: sign-ups per visitor for two onboarding flows, 100 visitors each. Flow A is drawn from a slightly higher mean (0.42) than flow B (0.37). Because these are *samples*, each group mean is fuzzy — quantifying that fuzziness is the whole point of the next step.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# A small real-ish A/B result: sign-ups per visitor, two onboarding flows.
rng = np.random.default_rng(0)
values_A = rng.normal(0.42, 0.10, 100)   # focus flow
values_B = rng.normal(0.37, 0.10, 100)   # context (baseline) flow

df = pd.DataFrame({
    "group": ["A"] * 100 + ["B"] * 100,
    "value": np.r_[values_A, values_B],
})

### Step 2 — Compute each group's mean and 95% confidence interval

A confidence interval turns a point estimate into a range that would contain the true mean about 95 times out of 100. The half-width is the standard error (`s / sqrt(n)`) scaled by the t-multiplier for 95% (about 1.96 for large `n`). We compute the mean, half-width, and sample size for each group so the chart can show how trustworthy each bar is.

In [ ]:
# Mean + 95% confidence interval per group (show the uncertainty).
def mean_ci(x, conf=0.95):
    x = np.asarray(x)
    n = len(x)
    m = x.mean()
    se = x.std(ddof=1) / np.sqrt(n)               # SE = s / sqrt(n)
    h = se * stats.t.ppf((1 + conf) / 2, n - 1)   # ~1.96 * SE for large n
    return m, h, n

stats_by_group = {g: mean_ci(d["value"]) for g, d in df.groupby("group")}
groups = list(stats_by_group)                     # ["A", "B"]
means  = [stats_by_group[g][0] for g in groups]
cis    = [stats_by_group[g][1] for g in groups]   # half-widths (the +/-)
ns     = [stats_by_group[g][2] for g in groups]

### Step 3 — Plot means with error bars and a finding-stating story

Three storytelling moves go on top of the bars. We **grey out** the baseline and highlight the focus group so attention lands in one place; we draw the 95% CI as error bars and annotate each `n`; and we give the chart a *finding-stating* title plus an annotation pointing at the disjoint intervals — so the reader gets the conclusion, not just the data.

In [ ]:
# Plot means WITH error bars; grey context, highlight focus.
focus = "A"
colors = ["#1f77b4" if g == focus else "#cccccc" for g in groups]  # grey vs highlight

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(groups, means, yerr=cis, capsize=8,
              color=colors, edgecolor="none",
              error_kw=dict(ecolor="#444", lw=1.5))

# Annotate each bar with its sample size n.
for g, m, h, n in zip(groups, means, cis, ns):
    ax.text(g, m + h + 0.005, f"n = {n}", ha="center", va="bottom",
            fontsize=9, color="#444")

# Finding-stating title (states the conclusion, not "Conversion by flow").
ax.set_title("Flow A converts ~5 points higher — 95% CIs don't overlap",
             fontsize=12, fontweight="bold", loc="left")
ax.set_ylabel("Sign-ups per visitor")
ax.set_ylim(0, max(m + h for m, h in zip(means, cis)) * 1.25)

# Annotate the key point, pointing at the focus bar.
fi = groups.index(focus)
ax.annotate("Real gap: intervals are disjoint",
            xy=(fi, means[fi] - cis[fi]), xytext=(fi + 0.15, means[fi] * 0.55),
            fontsize=9, color="#1f77b4",
            arrowprops=dict(arrowstyle="->", color="#1f77b4"))

ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, 0.01, "Error bars = 95% confidence interval", fontsize=8, color="#777")
plt.tight_layout()
plt.show()

## Visualize the data & results

_Real wine chemistry: the mean 'proline' level differs by grape variety — but by how much, and is the gap real? Bare means vs means with 95% confidence intervals._

### Step 1 — Load real wine data and pick a feature

We swap the toy A/B data for 178 real wines with 13 chemistry measurements each. We focus on `proline`, an amino acid whose level depends strongly on grape variety, and group the wines by their variety (`target`).

In [ ]:
import numpy as np
from sklearn.datasets import load_wine
from scipy import stats

d = load_wine(as_frame=True)
df = d.frame                    # 178 real wines, 13 chemistry features
feat = "proline"               # strongly variety-dependent

### Step 2 — Compute a 95% CI per variety and check overlap

For each variety we compute the mean `proline` and its 95% confidence interval, using the same standard-error formula as before. The payoff: the three intervals do not overlap, so the differences between varieties are real signal, not sampling noise — exactly the judgment error bars are meant to support.

In [ ]:
rows = []
for cls in sorted(df["target"].unique()):
    x  = df.loc[df["target"] == cls, feat].values
    n  = len(x)
    m  = x.mean()
    se = x.std(ddof=1) / np.sqrt(n)       # SE = s / sqrt(n)
    h  = se * stats.t.ppf(0.975, n - 1)   # 95% CI half-width (~1.96*SE)
    rows.append((cls, n, m, h, m - h, m + h))

for cls, n, m, h, lo, hi in rows:
    print(f"class {cls}: n={n}  mean={m:7.1f}  95% CI = [{lo:6.1f}, {hi:6.1f}]  (+/- {h:.1f})")

# class 0: n=59  mean=1115.7  95% CI = [1058.0, 1173.4]  (+/- 57.7)
# class 1: n=71  mean= 519.5  95% CI = [ 482.3,  556.7]  (+/- 37.2)
# class 2: n=48  mean= 629.9  95% CI = [ 596.5,  663.3]  (+/- 33.4)
# The intervals don't overlap -> the class differences are real, not sampling noise.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Your slide shows two bars: model X at 0.84 accuracy and model Y at 0.82, no error bars, titled "Accuracy by model". List three things wrong and fix each.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Add uncertainty: compute and draw a 95% CI (or SE) on each bar, and state $n$ (the test-set size). — _Bare point estimates overclaim; a 0.02 gap may be inside the noise. The interval lets the reader judge if X truly beats Y._
- Label what the error bars are once they exist: "error bars = 95% CI". — _An unlabeled bar is ambiguous &mdash; SD, SE, and CI are different sizes and imply different things._
- Rewrite the title to state the finding, e.g. "Models X and Y are within noise (overlapping 95% CIs, n = 2k)". — _A descriptive title makes the reader do the work; a finding-stating title delivers the conclusion._

**Answer:** (1) No uncertainty &mdash; add 95% CIs and the test-set $n$; a 0.02 gap is likely noise. (2) Unlabeled bars &mdash; caption them ("95% CI"). (3) Descriptive title &mdash; replace "Accuracy by model" with the actual conclusion, e.g. that the two are within noise. Bonus: grey one series if there is a focus, and don't cherry-pick the one test split where X happened to win.

</details>

**Problem 2.** A conversion rate of 60% comes from one segment with 5 users and another with 5,000 users. Both get the same fat bar on your chart. What is the problem and the fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that the standard error scales as $1/\sqrt{n}$, so the 5-user estimate is far less certain than the 5,000-user one. — _$\mathrm{SE}=s/\sqrt{n}$: with $n=5$ the bar is about $\sqrt{1000}\approx32$ times wider than with $n=5{,}000$._
- Draw each bar with its own 95% CI and annotate $n$ on each. — _The 5-user bar should be visibly enormous; the 5,000-user bar tight. Equal-looking bars hide that one number is a guess._
- Resist concluding anything from the 5-user segment. — _Its interval may span most of 0&ndash;100%; calling it "60%" as if exact is overclaiming on a tiny sample._

**Answer:** The chart ignores sample size: 60% from $n=5$ and 60% from $n=5{,}000$ carry wildly different certainty, but identical bars imply they're equally trustworthy. Fix: give each bar its real 95% CI (the $n=5$ bar will be huge because $\mathrm{SE}=s/\sqrt{n}$) and annotate $n$. Don't draw conclusions from the tiny segment.

</details>

**Problem 3.** You want to convince leadership that "Sign-ups doubled after the May launch" from a line chart of weekly sign-ups. Sketch the storytelling moves you'd make.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Put the conclusion in the title: "Sign-ups doubled after the May launch", not "Weekly sign-ups". — _The title is the first thing read; state the finding so the chart confirms it rather than asking the reader to deduce it._
- Structure context &rarr; insight &rarr; resolution: show the flat pre-launch baseline, mark the launch, then the rise. — _A clear arc makes the change feel like a story with a cause, not a random wiggle._
- Annotate the launch week with a vertical marker and a short label; grey the pre-launch line, highlight the post-launch line. — _One annotation and a single bright series direct attention to the exact moment that proves the point._
- Add a confidence band or note the sample size so the claim is hedged, and avoid cherry-picking a flattering window. — _Honesty: show the uncertainty and the full timeline so the doubling isn't an artifact of where you cropped._

**Answer:** Finding-stating title ("Sign-ups doubled after the May launch"); a context &rarr; insight &rarr; resolution arc (flat baseline, launch marker, rise); one annotation on the launch week; grey the baseline, highlight the post-launch line; and a confidence band / sample-size note with no cherry-picked window so the claim stays honest. One chart, one takeaway.

</details>